# Mô hình chính v2: `pair_skip_mlp_specfeat` (pairwise) so với `main_pairbce` (pairwise + BCE), kèm mô hình triển khai

**Settings → Accelerator:** GPU T4 x2 hoặc P100. GPU chỉ dùng để mã hóa BGE-M3; phần huấn luyện chạy trên CPU.
**Internet:** On.
**Không cần khóa API:** kết quả trích xuất bằng LLM đã có sẵn trong `data/cache/req_cache.json`.

Bấm **Save Version → Save & Run All**. Bản v1 (một cấu hình) chạy hết khoảng 1 giờ 15 phút trên T4 x2; bản này chạy 2 cấu hình nên ước tính khoảng 2,5–3 giờ.

Notebook làm các việc sau:
- **Mô hình triển khai:** `train_req.py final` (`main_pairbce`) → `req_model.pt` và ngưỡng mức 0–3.
- **Đánh giá 3 thiết lập, mỗi thiết lập 5 seed** (cùng fold, validation và seed với báo cáo), cho cả 2 cấu hình:
  1. Bộ đề tài (1.664 cặp), 5 fold chia theo JD.
  2. Bộ đề tài, loại trừ cả hai chiều CV và JD (10 ô).
  3. resume-job-fit (7.993 cặp), 5 fold chia theo JD.

Ô gần cuối in NDCG@10 trên tập kiểm tra theo từng cấu hình. Kết quả v1 của `pair_skip_mlp_specfeat`: 0,889 ± 0,007; 0,908 ± 0,009; 0,782 ± 0,003. Các file `*_kaggle2*.csv` ở tab Output dùng được trực tiếp cho `report/b_eval2.py` và `report/b_eval_ext.py`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc
!git clone -q -b new-update-version https://github.com/hoangphungvan/doantotnghiep.git
%cd doantotnghiep
!git log --oneline -1

In [ ]:
# Chỉ cài phần còn thiếu, KHÔNG cài lại torch (giữ bản CUDA của Kaggle)
!pip install -q torch_geometric python-dotenv openai pymupdf
import torch, sentence_transformers; print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__)

In [ ]:
# resume-job-fit không nằm trong repo: tải train.csv/test.csv gốc từ HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('cnamuangtoun/resume-job-description-fit', repo_type='dataset', local_dir='data/raw/cnamuangtoun')
!ls -la data/raw/cnamuangtoun

In [ ]:
# 0) Mô hình triển khai (main_pairbce): 5 fold theo JD để đo và fit ngưỡng mức 0-3, rồi huấn luyện lại trên toàn bộ
!python -X utf8 src/modeling/train_req.py final && cp models/req_model.pt /kaggle/working/

In [ ]:
CONFIGS = 'pair_skip_mlp_specfeat,main_pairbce'
# 1) Bộ đề tài, 5 fold chia theo JD, 5 seed
!python -X utf8 src/modeling/train_req.py jd --seeds 5 --configs {CONFIGS} --tag _kaggle2 && cp report/results/req_jd_kaggle2*.csv /kaggle/working/

In [ ]:
# 2) Bộ đề tài, loại trừ cả hai chiều (2 nhóm CV x 5 nhóm JD), 5 seed
!python -X utf8 src/modeling/train_req.py double --seeds 5 --configs {CONFIGS} --tag _kaggle2 && cp report/results/req_double_kaggle2*.csv /kaggle/working/

In [ ]:
# 3) resume-job-fit, 5 fold chia theo JD, 5 seed. Dòng 'train ... cặp' phải cộng lại 7993 cặp sau khi bỏ trùng. File được ghi sau mỗi seed
!python -X utf8 src/modeling/train_ext.py --protocol jd --seeds 5 --configs {CONFIGS} --tag _kaggle2; cp report/results/ext_cna_req_kaggle2*.csv /kaggle/working/

In [ ]:
TAG = EXT_TAG = '_kaggle2'
# NDCG@10 trên tập kiểm tra (trung bình ± độ lệch chuẩn theo seed) cho từng cấu hình, cùng cách tính với report/b_eval2.py, b_eval_ext.py
import sys, pandas as pd
sys.path.insert(0, '.')
from src.evaluation.metrics import evaluate_ranking
from src.modeling.train_ext import load
own = pd.read_csv('data/raw/pairs_labeled.csv')[['cv_id', 'jd_id', 'grade']]
d = load()
ext = pd.concat([d['train'], d['test']]).drop_duplicates(['cv_id', 'jd_id'])[['cv_id', 'jd_id', 'grade']]
for name, f, lab in (('Bộ đề tài, chia theo JD', 'req_jd' + TAG, own), ('Bộ đề tài, loại trừ hai chiều', 'req_double' + TAG, own), ('resume-job-fit, chia theo JD', 'ext_cna_req' + EXT_TAG, ext)):
    try:
        A = pd.read_csv(f'report/results/{f}.csv').merge(lab, on=['cv_id', 'jd_id'])
    except FileNotFoundError:
        print(name, ': chưa chạy'); continue
    for cfg in CONFIGS.split(','):
        L = A[A.config == cfg]
        if L.empty:
            continue
        elig = {q for q, x in L[L.seed == L.seed.min()].groupby('qid') if len(x) >= 2 and (x.grade >= 2).any()}
        r = pd.Series([evaluate_ranking({q: list(zip(y.grade, y.score)) for q, y in x[x.qid.isin(elig)].groupby('qid')}, ks=[10])['ndcg@10']
                       for _, x in L.groupby('seed')])
        print(f'{name} | {cfg}: NDCG@10 = {r.mean():.3f} ± {r.std():.3f} ({len(r)} seed, {len(elig)} truy vấn)')

In [ ]:
!cp report/results/*kaggle2*.csv /kaggle/working/ 2>/dev/null; ls -la /kaggle/working/*.csv /kaggle/working/*.pt